In [ ]:
# Library

import os 
from getpass import getpass
from dotenv import load_dotenv
import json
import requests
from datetime import datetime
from IPython.display import display, Markdown

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.memory import ConversationBufferMemory
from langchain.agents import create_tool_calling_agent, AgentExecutor, load_tools

load_dotenv()

True

In [3]:
# must enter API key 
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

# below should not be changed
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# you can change this as preferred
os.environ["LANGCHAIN_PROJECT"] = "damar-ananta-langsmith-openai"

In [4]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# For normal accurate responses
llm = ChatOpenAI(temperature=0.0, model="gpt-4o-mini")

# Introduction to Tools

In [ ]:
""" 
Apa itu Tool?
Tool adalah fungsi yang diberikan kepada LLM agar LLM dapat melakukan suatu tindakan menggunakan program atau layanan eksternal.

Contohnya:
- Kalkulator
- Web search
- Mengakses API
- Membaca database
- Mengambil lokasi
- Membaca tanggal dan waktu
LLM sendiri merupakan model yang menghasilkan teks. Dengan Tool, LLM bisa meminta sistem menjalankan suatu fungsi.

Komponen penting Tool
Ketika membuat Tool, notebook menyarankan:
1. Docstring — menjelaskan apa fungsi Tool dan kapan digunakan.
2. Parameter yang jelas — nama parameter harus mudah dipahami LLM.
3. Type annotation — memberikan informasi tipe input dan output.


@tool membuat fungsi Python menjadi objek StructuredTool yang memiliki:
- name
- description
- args_schema
- func
Schema ini membantu LLM mengetahui parameter yang perlu diberikan.
"""

In [7]:
@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x

In [ ]:
#Melihat metadata Tool

add

StructuredTool(name='add', description="Add 'x' and 'y'.", args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x10b37c4a0>)

In [9]:
print(f"{add.name=}\n{add.description=}")

add.name='add'
add.description="Add 'x' and 'y'."


In [10]:
add.args_schema.model_json_schema()

{'description': "Add 'x' and 'y'.",
 'properties': {'x': {'title': 'X', 'type': 'number'},
  'y': {'title': 'Y', 'type': 'number'}},
 'required': ['x', 'y'],
 'title': 'add',
 'type': 'object'}

In [11]:
exponentiate.args_schema.model_json_schema()

{'description': "Raise 'x' to the power of 'y'.",
 'properties': {'x': {'title': 'X', 'type': 'number'},
  'y': {'title': 'Y', 'type': 'number'}},
 'required': ['x', 'y'],
 'title': 'exponentiate',
 'type': 'object'}

In [ ]:
""" 
- Schema akan menunjukkan parameter x dan y, keduanya bertipe number dan required.
- Hal ini memungkinkan LLM mengirim argument dalam format terstruktur, misalnya:
"""

llm_output_string = "{\"x\": 5, \"y\": 2}"  # this is the output from the LLM
llm_output_dict = json.loads(llm_output_string)  # load as dictionary
llm_output_dict

{'x': 5, 'y': 2}

In [ ]:
# Kemudian dictionary tersebut dapat di-unpack menjadi keyword arguments

exponentiate.func(**llm_output_dict)

25

# Creating an Agent

In [ ]:
""" 
- Sekarang Tool sudah tersedia. Kita perlu menggabungkannya dengan LLM untuk membuat Agent.
- Notebook menggunakan konsep LCEL (LangChain Expression Language) untuk menyusun Agent.
"""

In [ ]:
# Membuat Prompt

prompt = ChatPromptTemplate.from_messages([
    ("system", "you're a helpful assistant"), # Memberikan instruksi dasar kepada LLM.
    MessagesPlaceholder(variable_name="chat_history"), #Bagian ini merupakan tempat memasukkan riwayat percakapan sebelumnya.
    ("human", "{input}"), # {input} adalah placeholder untuk pertanyaan User.
    ("placeholder", "{agent_scratchpad}"), # agent_scratchpad menyimpan proses kerja Agent dalam satu task, terutama Tool calls dan hasilnya.
])

""" 
Ini merupakan tempat menyimpan intermediate steps Agent.

Contohnya, ketika Agent sedang menyelesaikan perhitungan:
1. Memutuskan menggunakan add.
2. Tool add dijalankan.
3. Mendapatkan hasil 19.
4. Memutuskan menggunakan multiply.
5. Tool multiply dijalankan.
6. Mendapatkan hasil.
7. Menyusun jawaban akhir.
Informasi dari langkah Tool sebelumnya dibutuhkan Agent agar bisa melanjutkan prosesnya.
chat_history menyimpan percakapan User dan Assistant, sedangkan agent_scratchpad menyimpan proses kerja Agent dalam satu task, terutama Tool calls dan hasilnya.

"""

In [ ]:
# Menambahkan Chat Memory

memory = ConversationBufferMemory( # Ini adalah memory yang menyimpan percakapan sebelumnya.
    memory_key="chat_history",  # Nama variabel memory harus sesuai dengan placeholder dalam prompt
    return_messages=True  # Memory mengembalikan history dalam bentuk objek Message LangChain.
)

/var/folders/sd/7_dvq1bs4x12529lgl1190_w0000gn/T/ipykernel_22333/1439794951.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(
/usr/local/lib/python3.11/site-packages/pydantic/main.py:214: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [ ]:
# Membuat Agent

from langchain.agents import create_tool_calling_agent

tools = [add, subtract, multiply, exponentiate] #menggabungkan Tools menjadi list

agent = create_tool_calling_agent(
    llm=llm, tools=tools, prompt=prompt
)

""" 
Apa yang dilakukan create_tool_calling_agent?
Fungsi ini menggabungkan:
- LLM
- Daftar Tools
- Prompt
Menjadi sebuah Agent yang dapat memilih Tool dan menghasilkan Tool Call.

Namun, poin yang sangat penting:
agent belum menjalankan Tool secara otomatis sampai execution loop dijalankan.
"""

In [ ]:
agent.invoke({ #Hasilnya bukan jawaban final, tetapi sebuah ToolAgentAction.
    "input": "what is 10.7 multiplied by 7.68?",
    "chat_history": memory.chat_memory.messages,
    "intermediate_steps": []  # agent will append it's internal steps here
})

""" 
Artinya LLM telah melakukan hal berikut:
1. Memahami pertanyaan User.
2. Menentukan bahwa operasi yang dibutuhkan adalah perkalian.
3. Memilih Tool multiply.
4. Menyiapkan argument x=10.7 dan y=7.68.
Namun, Tool belum dieksekusi.
"""

In [ ]:
# Agent Execution Loop
agent_executor = AgentExecutor( #Untuk menjalankan Tool dan mengembalikan hasilnya kepada LLM, notebook menggunakan AgentExecutor.
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True
)

""" 
Parameter
- agent=agent
Agent yang sudah dibuat sebelumnya.
- tools=tools
Daftar Tool yang dapat dieksekusi oleh Executor.
- memory=memory
Memory yang digunakan untuk menyimpan percakapan.
- verbose=True
Menampilkan proses internal Agent di output.
"""

In [ ]:
agent_executor.invoke({
    "input": "what is 10.7 multiplied by 7.68?",
    "chat_history": memory.chat_memory.messages,
})

""" 
Perhatikan bahwa pada AgentExecutor, kita tidak perlu lagi memasukkan:
    "intermediate_steps": []

- Karena Agent Executor mengelola intermediate steps secara internal.
"""

In [ ]:
10.7*7.68

In [ ]:
# Agent menggunakan Memory

agent_executor.invoke({
    "input": "My name is James",
    "chat_history": memory #menguji apakah Agent bisa mengingat nama User.
})

""" 
Mengapa Agent bisa mengingatnya?
Karena history percakapan sebelumnya diberikan kembali ke Agent melalui chat_history.
"""

In [ ]:
""" 
Hubungan Memory dengan Agent:
- Agent tidak otomatis mempunyai memory permanen.
- Kemampuan mengingat bergantung pada apakah conversation history diberikan kembali kepada LLM.
"""

In [ ]:
agent_executor.invoke({
    "input": "What is nine plus 10, minus 4 * 2, to the power of 3",
    "chat_history": memory
})

In [ ]:
9+10-(4*2)**3

In [ ]:
""" 
Agent gagal menyelesaikan interpretasi dan perhitungan ini dengan benar. Ini contoh bagus bahwa Agent bukan berarti selalu benar, bahkan saat menggunakan Tools.

Tools membantu, tetapi LLM masih bisa:
- Memilih urutan operasi yang salah.
- Menginterpretasikan pertanyaan dengan salah.
- Memberikan argument yang tidak sesuai.
- Menggabungkan hasil Tool secara keliru.
"""

In [ ]:
agent_executor.invoke({
    "input": "What is my name",
    "chat_history": memory
})

# SepAPI Weather Agent

In [ ]:
os.environ["SERPAPI_API_KEY"] = os.getenv("SERPAPI_API_KEY") 

In [ ]:
toolbox = load_tools( # Digunakan untuk memuat Tool yang sudah disediakan oleh LangChain.
    tool_names=['serpapi'],  #Menentukan Tool yang ingin dimuat
    llm=llm
    )

In [ ]:
@tool
def get_location_from_ip():
    """Get the geographical location based on the IP address."""
    try:
        response = requests.get("https://ipinfo.io/json") #Untuk mengambil informasi lokasi berdasarkan IP. Mengapa lokasi menjadi Tool? Karena LLM tidak mengetahui secara otomatis lokasi IP User pada saat permintaan berlangsung. Dengan Tool ini, Agent dapat mengambil informasi lokasi terkini melalui API.
        data = response.json()
        if 'loc' in data:
            latitude, longitude = data['loc'].split(',') #Kemudian koordinat dipisahkan
            data = ( #Dan data dirangkai menjadi string
                f"Latitude: {latitude},\n"
                f"Longitude: {longitude},\n"
                f"City: {data.get('city', 'N/A')},\n"
                f"Country: {data.get('country', 'N/A')}"
            )
            return data
        else:
            return "Location could not be determined."
    except Exception as e:
        return f"Error occurred: {e}"

@tool
def get_current_datetime() -> str: # Tool ini mengambil tanggal dan waktu dari sistem komputer yang menjalankan Python.
    """Return the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "you're a helpful assistant"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}") # Scratchpad tetap dibutuhkan karena Agent dapat menggunakan beberapa Tool secara berurutan.
])

In [ ]:
tools = toolbox + [get_current_datetime, get_location_from_ip] # Notebook membuat list Tools

agent = create_tool_calling_agent( # Kemudian membuat Agent
    llm=llm, tools=tools, prompt=prompt
)

agent_executor = AgentExecutor( # Dan Executor
    agent=agent, tools=tools, verbose=True
)

In [ ]:
# Menjalankan Weather Agent
out = agent_executor.invoke({
    "input": (
        "I have a few questions, what is the date and time right now? "
        "How is the weather where I am? Please give me degrees in Celsius"
    )
})

In [ ]:
display(Markdown(out["output"]))

In [ ]:
""" 
LLM biasa
- User → Prompt → LLM → Jawaban
- Hanya menghasilkan respons berdasarkan konteks dan pengetahuan model.

Agent
- User → Agent → Memilih Tool → Eksekusi Tool → Observation → Agent → Jawaban
- Dapat melakukan tindakan dan menggunakan hasil tindakan tersebut dalam proses reasoning/task solvin
"""